# 06. Fatorial

Em vez de testar um fator de cada vez, o desenho **fatorial** varia vários fatores ao mesmo tempo e estima, de uma só vez, os **efeitos principais** e as **interações**. É aqui que a biblioteca encosta no DoE clássico (Fisher, Box).

> Atenção ao vocabulário: aqui "interação" é um **efeito estimado** (`A:B`), não o *interaction plot* de médias da otimização de processo.

## O experimento

Dois fatores binários, `A` e `B`. O resultado depende de cada um e também da combinação (há interação). O `FactorialDesign` exige `n_per_cell * 2^K` unidades; com `K=2` e `n_per_cell=100`, são 400.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import FactorialAssignment
from skxperiments.design.factorial import FactorialDesign

rng = np.random.default_rng(13)
n_per_cell = 100
n = n_per_cell * 4

design = FactorialDesign(factors=["A", "B"], n_per_cell=n_per_cell, seed=13)
assignment = design.randomize(pd.DataFrame({"unit": np.arange(n)}))

A = assignment.data_["A"].to_numpy()
B = assignment.data_["B"].to_numpy()
y = 2.0 * A + 1.0 * B + 1.0 * (A * B) + rng.normal(size=n) * 0.3

data = assignment.data_.copy()
data["y"] = y
assignment = FactorialAssignment(
    data=data,
    design=design,
    factor_cols=assignment.factor_cols,
    cell_sizes=assignment.cell_sizes_,
    seed=13,
)

## Estimar todos os efeitos

`FactorialEstimator` devolve os `2^K - 1` contrastes: efeitos principais (`A`, `B`) e a interação (`A:B`).

In [ ]:
from skxperiments.estimators.factorial_estimator import FactorialEstimator

result = FactorialEstimator(outcome_col="y").fit(assignment).estimate()
for key, value in result.effects.items():
    print(f"{':'.join(key)}: {value:.3f}")

Os efeitos principais de `A` e `B` aparecem claramente, e o termo `A:B` é não nulo: como há interação, o efeito conjunto difere da soma dos efeitos isolados.

## Visualizar

`plot_interaction` mostra os efeitos lado a lado (precisa de matplotlib, no extra `viz`).

In [ ]:
from skxperiments.reporting import plot_interaction

ax = plot_interaction(result)
ax.figure

## O que aprendemos

- Um único desenho fatorial estima efeitos principais e interações de vários fatores.
- A interação `A:B` captura o quanto o efeito de um fator depende do nível do outro.
- `FactorialEstimator` devolve só os pontos; inferência sobre efeitos fatoriais fica para a v2.

**Próximo:** `07. Muitos testes` mostra como corrigir p-valores quando há vários efeitos ou experimentos.